In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q "/content/drive/MyDrive/data/wikipedia.txt.zip" -d "/content/wikipedia.txt"

- 16GB
- On average each word has 10 letters so uses 10B
- So on average the file has 1'600'000'000 words
- Larger word might be 15, so it could exist 26^16=4.3608743e+22 words
- So it might be needed 1.7443497e+23 B to count frequency of all the words

In [4]:
import os
import multiprocessing

print("=== CPU core information ===")
print("os.cpu_count():", os.cpu_count())
print("multiprocessing.cpu_count():", multiprocessing.cpu_count())

try:
    with open('/proc/cpuinfo', 'r') as f:
        cpuinfo = f.read()
    print("/proc/cpuinfo processor count:", cpuinfo.count('processor\t:'))
except Exception as e:
    print("Unable to read /proc/cpuinfo:", e)

print("\n=== Multicore program note ===")
print("This Colab kernel can see the number of CPU cores above.")
print("You can write multicore programs in Colab using multiprocessing or process-based parallelism.")
print("Pure Python threads are still limited by the GIL for CPU-bound work, so use multiprocessing for true parallel CPU work.")
print("GPU or CUDA code is a different form of parallelism and works if the runtime provides a GPU.")

=== CPU core information ===
os.cpu_count(): 2
multiprocessing.cpu_count(): 2
/proc/cpuinfo processor count: 2

=== Multicore program note ===
This Colab kernel can see the number of CPU cores above.
You can write multicore programs in Colab using multiprocessing or process-based parallelism.
Pure Python threads are still limited by the GIL for CPU-bound work, so use multiprocessing for true parallel CPU work.
GPU or CUDA code is a different form of parallelism and works if the runtime provides a GPU.


In [5]:
import os

inPath = "/content/wikipedia.txt/wikipedia.txt"

if os.path.exists(inPath):
    # Get total file size for reference
    file_size_gb = os.path.getsize(inPath) / (1024**3)
    
    # Read a 1MB chunk (1,048,576 bytes)
    # Using 'errors=ignore' handles potential non-UTF8 characters in raw dumps
    with open(inPath, 'r', encoding='utf-8', errors='ignore') as f:
        chunk = f.read(100000000) 
        
    print(f"File found! Total Size: {file_size_gb:.2f} GB")
    print("-" * 40)
    print("--- FIRST 2000 CHARACTERS OF CHUNK ---")
    print(chunk[:200000]) 
else:
    print(f"File not found at: {inPath}")
    print("Check if you unzipped it correctly or if the folder name is duplicated.")


File not found at: /content/wikipedia.txt/wikipedia.txt
Check if you unzipped it correctly or if the folder name is duplicated.


In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpdude_f16".


In [ ]:
%%cuda --compiler-args=-gencode=arch=compute_75,code=sm_75
#include <iostream>
#include <math.h>



In [ ]:
%%cuda --compiler-args=-gencode=arch=compute_75,code=sm_75
#include <iostream>
#include <math.h>

__global__
void matrixAddKernel(float* C, float* A, float* B, int n) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < n*n) {
        C[i] = A[i] + B[i];
    }
}
__global__
void matrixAddKernelByRow(float* C, float* A, float* B, int n) {
    int row = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < n) {
        for (int col = 0; col < n; ++col) {
            int i = row * n + col;
            C[i] = A[i] + B[i];
        }
    }
}
__global__
void matrixAddKernelByCol(float* C, float* A, float* B, int n) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (col < n) {
        for (int row = 0; row < n; ++row) {
            int i = row * n + col;
            C[i] = A[i] + B[i];
        }
    }
}

void matrixAdd(float* C, float* A, float* B, int n) {
    int size = n*n * sizeof(float);
    float *dA, *dB, *dC;

    cudaMalloc((void **) &dA, size);
    cudaMemcpy(dA, A, size, cudaMemcpyHostToDevice);
    cudaMalloc((void **) &dB, size);
    cudaMemcpy(dB, B, size, cudaMemcpyHostToDevice);
    cudaMalloc((void **) &dC, size);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    matrixAddKernel<<<ceil(n*n/1024.0), 1024>>>(dC, dA, dB, n);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Kernel execution time: %f ms\n", milliseconds);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaMemcpy(C, dC, size, cudaMemcpyDeviceToHost);
    cudaFree(dA); cudaFree(dB); cudaFree(dC);
}
void matrixAddByRow(float* C, float* A, float* B, int n) {
    int size = n*n * sizeof(float);
    float *dA, *dB, *dC;

    cudaMalloc((void **) &dA, size);
    cudaMemcpy(dA, A, size, cudaMemcpyHostToDevice);
    cudaMalloc((void **) &dB, size);
    cudaMemcpy(dB, B, size, cudaMemcpyHostToDevice);
    cudaMalloc((void **) &dC, size);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    matrixAddKernelByRow<<<ceil(n/1024.0), 1024>>>(dC, dA, dB, n);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Row Kernel execution time: %f ms\n", milliseconds);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaMemcpy(C, dC, size, cudaMemcpyDeviceToHost);
    cudaFree(dA); cudaFree(dB); cudaFree(dC);
}
void matrixAddByCol(float* C, float* A, float* B, int n) {
    int size = n*n * sizeof(float);
    float *dA, *dB, *dC;

    cudaMalloc((void **) &dA, size);
    cudaMemcpy(dA, A, size, cudaMemcpyHostToDevice);
    cudaMalloc((void **) &dB, size);
    cudaMemcpy(dB, B, size, cudaMemcpyHostToDevice);
    cudaMalloc((void **) &dC, size);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    matrixAddKernelByCol<<<ceil(n/1024.0), 1024>>>(dC, dA, dB, n);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Col Kernel execution time: %f ms\n", milliseconds);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaMemcpy(C, dC, size, cudaMemcpyDeviceToHost);
    cudaFree(dA); cudaFree(dB); cudaFree(dC);
}

int main() {
    int n = 1000;
    size_t bytes = n*n * sizeof(float);

    float* hA = (float*)malloc(bytes);
    float* hB = (float*)malloc(bytes);
    float* hC = (float*)malloc(bytes);

    for(int i = 0; i < n*n; i++) {
        hA[i] = sinf(i) * sinf(i);
        hB[i] = cosf(i) * cosf(i);
    }

    matrixAdd(hC, hA, hB, n);
    matrixAddByRow(hC, hA, hB, n);
    matrixAddByCol(hC, hA, hB, n);

    // Verify result
    float maxError = 0.0f;
    for(int i = 0; i < n*n; i++) {
        maxError = fmax(maxError, fabs(hC[i] - 1.0f));
    }
    printf("Max error: %f\n", maxError);

    free(hA); free(hB); free(hC);
    return 0;
}

In [2]:
import subprocess
output = subprocess.run(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True
)
print(output.stdout)

FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

In [3]:
%%cuda --compiler-args=-gencode=arch=compute_75,code=sm_75
#include <iostream>
#include <math.h>

int main(){
  int dev_count;
  cudaGetDeviceCount(&dev_count);
  printf("Number of devices: %d\n", dev_count);

  cudaDeviceProp dev_prop;
  cudaGetDeviceProperties(&dev_prop, 0);
  printf("Device name: %s\n", dev_prop.name);
  printf("Compute capability: %d.%d\n", dev_prop.major, dev_prop.minor);
  printf("Total global memory: %.2f GB\n", dev_prop.totalGlobalMem / (1024.0 * 1024.0 * 1024.0));
  printf("Shared memory per block: %zu bytes\n", dev_prop.sharedMemPerBlock);
  printf("Registers per block: %d\n", dev_prop.regsPerBlock);
  printf("Warp size: %d\n", dev_prop.warpSize);
  printf("Max threads per block: %d\n", dev_prop.maxThreadsPerBlock);
  printf("Max grid dimensions: (%d, %d, %d)\n", dev_prop.maxGridSize[0], dev_prop.maxGridSize[1], dev_prop.maxGridSize[2]);
  printf("Max block dimensions: (%d, %d, %d)\n", dev_prop.maxThreadsDim[0], dev_prop.maxThreadsDim[1], dev_prop.maxThreadsDim[2]);
  printf("Clock rate: %.0f MHz\n", dev_prop.clockRate * 1e-3f);
  printf("Multi-processor count: %d\n\n", dev_prop.multiProcessorCount);
  printf("Max Threads per SM: %d\n", dev_prop.maxThreadsPerMultiProcessor);
  printf("Max Blocks per SM: %d\n", dev_prop.maxBlocksPerMultiProcessor);

  return 0;
}

FileNotFoundError: [Errno 2] No such file or directory: 'nvcc'